# 📊 E-Commerce Sales Analysis / E-Commerce Verkaufsanalyse
**Author / Autor:** Yurii Oleschuk  
**Dataset / Datensatz:** UCI Online Retail (541,909 transactions · Dec 2010 – Dec 2011)  
**Stack:** Python · SQL (SQLite) · Power BI  

---
### 🇬🇧 Business Questions
1. Which products and customers drive the most revenue?
2. How does customer retention change over time (cohort analysis)?
3. What are the key customer segments (RFM)?
4. When do customers buy most — and how can that inform campaign timing?
5. What is the sales forecast for next quarter?

### 🇩🇪 Geschäftsfragen
1. Welche Produkte und Kunden generieren den größten Umsatz?
2. Wie verändert sich die Kundenbindung über die Zeit (Kohortenanalyse)?
3. Welche Kundensegmente gibt es (RFM)?
4. Wann kaufen Kunden am meisten — und wie beeinflusst das die Kampagnenplanung?
5. Wie sieht die Umsatzprognose für das nächste Quartal aus?

## 1. Setup & Data Loading / Einrichtung & Datenladen

In [ ]:
# EN Import libraries
# DE Bibliotheken importieren
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# EN Style settings / DE Stileinstellungen
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# EN Load raw dataset / DE Rohdaten laden
df_raw = pd.read_csv('../data/sales.csv', encoding='ISO-8859-1')
print(f'Raw dataset / Rohdatensatz: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()

## 2. Data Cleaning & Quality Report / Datenbereinigung & Qualitätsbericht

In [ ]:
# EN Data quality overview before cleaning
# DE Datenqualität vor der Bereinigung
print('=== DATA QUALITY REPORT / DATENQUALITÄTSBERICHT ===')
print('\nMissing values / Fehlende Werte:')
print(df_raw.isnull().sum())
print(f"\nNegative Quantity (returns / Rücksendungen): {(df_raw['Quantity'] < 0).sum():,}")
print(f"Zero/Negative UnitPrice / Ungültige Preise:  {(df_raw['UnitPrice'] <= 0).sum():,}")
print(f"Missing CustomerID / Fehlende Kunden-ID:     {df_raw['CustomerID'].isnull().sum():,}")

In [ ]:
df = df_raw.copy()

# EN Remove returns and invalid prices
# DE Rücksendungen und ungültige Preise entfernen
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# EN Remove non-product entries (postage, manual charges etc.)
# DE Nicht-Produkteinträge entfernen (Porto, manuelle Gebühren usw.)
exclude_keywords = ['POSTAGE', 'DOTCOM', 'MANUAL', 'BANK CHARGES', 'AMAZONFEE', 'CRUK']
pattern = '|'.join(exclude_keywords)
df = df[~df['Description'].str.upper().str.contains(pattern, na=False)]

# EN Drop rows without CustomerID
# DE Zeilen ohne Kunden-ID entfernen
df = df.dropna(subset=['CustomerID'])
df = df.drop_duplicates()

# EN Type conversions / DE Datentypkonvertierungen
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID']  = df['CustomerID'].astype(int)

# EN Feature engineering / DE Neue Spalten erstellen
df['Revenue']   = df['Quantity'] * df['UnitPrice']
df['Month']     = df['InvoiceDate'].dt.to_period('M')
df['YearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Hour']      = df['InvoiceDate'].dt.hour

removed = df_raw.shape[0] - df.shape[0]
print(f'Clean dataset / Bereinigter Datensatz: {df.shape[0]:,} rows')
print(f'Removed / Entfernt: {removed:,} ({removed/df_raw.shape[0]:.1%})')
df.head()

## 3. KPI Dashboard

In [ ]:
# EN Calculate key performance indicators
# DE Kennzahlen berechnen
total_revenue   = df['Revenue'].sum()
total_orders    = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
aov             = total_revenue / total_orders
avg_freq        = df.groupby('CustomerID')['InvoiceNo'].nunique().mean()
repeat_rate     = (df.groupby('CustomerID')['InvoiceNo'].nunique() > 1).mean()

kpis = {
    'Total Revenue\nGesamtumsatz':               f'£{total_revenue:,.0f}',
    'Total Orders\nBestellungen gesamt':         f'{total_orders:,}',
    'Unique Customers\nEinzigartige Kunden':     f'{total_customers:,}',
    'Avg Order Value\nDurchschn. Bestellwert':   f'£{aov:,.2f}',
    'Avg Orders/Customer\nBest./Kunde':          f'{avg_freq:.1f}',
    'Repeat Customer Rate\nWiederkehrquote':     f'{repeat_rate:.1%}',
}

fig, axes = plt.subplots(2, 3, figsize=(14, 5))
for ax, (label, value) in zip(axes.flatten(), kpis.items()):
    ax.text(0.5, 0.62, value, ha='center', va='center', fontsize=22,
            fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center', fontsize=9,
            color='gray', transform=ax.transAxes, linespacing=1.5)
    ax.axis('off')
    ax.set_facecolor('#f8f9fa')
    for spine in ax.spines.values(): spine.set_visible(True)

fig.suptitle('Key Performance Indicators / Kennzahlen', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n🇬🇧 Insight: Repeat customer rate is {repeat_rate:.1%} — strong product-market fit.')
print(f'🇩🇪 Erkenntnis: Wiederkehrquote beträgt {repeat_rate:.1%} — starke Produkt-Markt-Passung.')

## 4. Revenue Trend & Seasonality / Umsatztrend & Saisonalität

In [ ]:
# EN Monthly aggregation / DE Monatliche Aggregation
monthly = df.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].bar(monthly['YearMonth'], monthly['Revenue'], color='steelblue', alpha=0.85)
axes[0].set_ylabel('Revenue / Umsatz (£)')
axes[0].set_title('Monthly Revenue / Monatlicher Umsatz', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))

axes[1].plot(monthly['YearMonth'], monthly['Orders'], marker='o', color='darkorange')
axes[1].set_ylabel('Orders / Bestellungen')
axes[1].set_title('Monthly Orders / Monatliche Bestellungen', fontweight='bold')

axes[2].plot(monthly['YearMonth'], monthly['Customers'], marker='s', color='green')
axes[2].set_ylabel('Customers / Kunden')
axes[2].set_title('Monthly Active Customers / Monatl. aktive Kunden', fontweight='bold')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Sales Trends 2010-2011 / Verkaufstrends 2010-2011', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

peak_month = monthly.loc[monthly['Revenue'].idxmax(), 'YearMonth']
peak_rev   = monthly['Revenue'].max()
print(f'\n🇬🇧 Insight: Peak month is {peak_month} (£{peak_rev:,.0f}). Clear Q4 seasonality — Christmas gifting.')
print('   Recommendation: Front-load inventory and increase ad spend in Oct-Nov.')
print(f'\n🇩🇪 Erkenntnis: Stärkster Monat ist {peak_month} (£{peak_rev:,.0f}). Q4-Saisonalität — Weihnachtsgeschäft.')
print('   Empfehlung: Lagerbestand im Oktober/November aufbauen, Werbebudget erhöhen.')

## 5. Sales Heatmap — Day x Hour / Umsatz-Heatmap — Tag x Stunde

In [ ]:
# EN Revenue by day of week and hour / DE Umsatz nach Wochentag und Stunde
day_labels   = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heatmap_data = df.groupby(['DayOfWeek', 'Hour'])['Revenue'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Revenue / Umsatz (£)'})
ax.set_yticklabels([day_labels[i] for i in heatmap_data.index], rotation=0)
ax.set_xlabel('Hour of Day / Stunde')
ax.set_ylabel('Day of Week / Wochentag')
ax.set_title('Revenue Heatmap: Day x Hour / Umsatz-Heatmap: Tag x Stunde',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

peak_day  = day_labels[heatmap_data.sum(axis=1).idxmax()]
peak_hour = heatmap_data.sum(axis=0).idxmax()
print(f'\n🇬🇧 Insight: Peak buying window — {peak_day}, {peak_hour}:00-{peak_hour+1}:00. B2B pattern.')
print('   Recommendation: Schedule email campaigns for Tue-Thu mornings.')
print(f'\n🇩🇪 Erkenntnis: Spitzenkaufzeit — {peak_day}, {peak_hour}:00-{peak_hour+1}:00. B2B-Muster.')
print('   Empfehlung: E-Mail-Kampagnen dienstags bis donnerstags vormittags planen.')

## 6. Top Products & Pareto Analysis / Top-Produkte & Pareto-Analyse

In [ ]:
# EN Top products by revenue with Pareto curve
# DE Top-Produkte nach Umsatz mit Pareto-Kurve
top_products = (
    df.groupby('Description')
    .agg(Revenue=('Revenue','sum'), Quantity=('Quantity','sum'), Orders=('InvoiceNo','nunique'))
    .sort_values('Revenue', ascending=False)
    .head(10).reset_index()
)

product_rev    = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False)
cumulative_pct = product_rev.cumsum() / product_rev.sum() * 100
top_80_count   = (cumulative_pct <= 80).sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = sns.color_palette('Blues_r', n_colors=10)
axes[0].barh(top_products['Description'][::-1], top_products['Revenue'][::-1], color=colors)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
axes[0].set_title('Top 10 Products by Revenue\nTop-10-Produkte nach Umsatz', fontweight='bold')
axes[0].set_xlabel('Revenue / Umsatz (£)')

top_n = 30
axes[1].bar(range(top_n), product_rev.head(top_n).values, color='steelblue', alpha=0.7)
ax2 = axes[1].twinx()
ax2.plot(range(top_n), cumulative_pct.head(top_n).values, color='red', marker='.', lw=2)
ax2.axhline(80, color='red', linestyle='--', alpha=0.5)
ax2.set_ylabel('Cumulative % / Kumulativer Anteil %', color='red')
ax2.set_ylim(0, 105)
axes[1].set_title('Pareto Analysis / Pareto-Analyse', fontweight='bold')
axes[1].set_xlabel('Product Rank / Produktrang')

plt.tight_layout()
plt.show()

print(f'\n🇬🇧 Insight (Pareto): Top {top_80_count} products generate 80% of revenue. Focus inventory on core SKUs.')
print(f'🇩🇪 Erkenntnis (Pareto): Top {top_80_count} Produkte generieren 80% des Umsatzes. Lager auf Kernprodukte fokussieren.')
print('\nTop 10 Products / Top-10-Produkte:')
print(top_products[['Description','Revenue','Quantity','Orders']].to_string(index=False))

## 7. Cohort Retention Analysis / Kohortenanalyse zur Kundenbindung

In [ ]:
# EN Assign each customer their acquisition cohort (first purchase month)
# DE Jedem Kunden den Akquisitionsmonat zuweisen (erster Kauf)
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')
df['CohortIndex'] = (df['Month'] - df['CohortMonth']).apply(lambda x: x.n)

cohort_data  = df.groupby(['CohortMonth','CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_pct   = cohort_pivot.divide(cohort_pivot[0], axis=0).round(3) * 100

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(cohort_pct.iloc[:, :12], ax=ax, annot=True, fmt='.0f',
            cmap='RdYlGn', vmin=0, vmax=50, linewidths=0.3,
            cbar_kws={'label': 'Retention % / Bindungsrate %'})
ax.set_title('Monthly Cohort Retention (%) / Monatliche Kohortenretention (%)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase / Monate seit Erstkauf')
ax.set_ylabel('Cohort / Kohorte')
plt.tight_layout()
plt.show()

avg_m1 = cohort_pct[1].mean()
avg_m3 = cohort_pct[3].mean()
print(f'\n🇬🇧 Insight: Month-1 avg retention: {avg_m1:.1f}%. Only 1 in 4 customers return.')
print('   Recommendation: Launch 30-day post-purchase re-engagement email sequence.')
print(f'\n🇩🇪 Erkenntnis: Monat-1-Ø-Bindungsrate: {avg_m1:.1f}%. Nur 1 von 4 Kunden kehrt zurück.')
print('   Empfehlung: 30-Tage-E-Mail-Reaktivierungssequenz nach dem Kauf starten.')

## 8. RFM Customer Segmentation / RFM-Kundensegmentierung

In [ ]:
# EN Recency · Frequency · Monetary segmentation
# DE Aktualität · Häufigkeit · Geldwert — Kundensegmentierung
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate',  lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo',  'nunique'),
    Monetary=('Revenue',     'sum')
).reset_index()

# EN Score each dimension 1-4 (4 = best) / DE Bewertung 1-4 (4 = bester Wert)
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

def rfm_segment(row):
    r, f = int(row['R_score']), int(row['F_score'])
    if r >= 4 and f >= 4:   return 'Champions'
    elif r >= 3 and f >= 3: return 'Loyal Customers'
    elif r >= 4:             return 'Recent Customers'
    elif r >= 3:             return 'Potential Loyalists'
    elif r == 2 and f >= 2: return 'At Risk'
    elif r <= 2 and f <= 2: return 'Lost'
    else:                   return 'Need Attention'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)

seg_summary = rfm.groupby('Segment').agg(
    Customers=('CustomerID','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean'),
    Total_Revenue=('Monetary','sum')
).round(1).sort_values('Total_Revenue', ascending=False)

print('RFM Segment Summary / RFM-Segmentübersicht:')
print(seg_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = sns.color_palette('Set2', len(seg_summary))
seg_counts = rfm['Segment'].value_counts()
axes[0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
            startangle=140, colors=palette)
axes[0].set_title('Customer Segments\nKundensegmente', fontweight='bold')

seg_rev = seg_summary['Total_Revenue'].sort_values()
axes[1].barh(seg_rev.index, seg_rev.values, color=palette)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x/1000:.0f}k'))
axes[1].set_title('Revenue by Segment\nUmsatz nach Segment', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight:')
print('   Champions      -> protect with VIP loyalty programme')
print('   At Risk        -> win-back campaign with personalised discount')
print('   Lost           -> deprioritise, re-acquisition cost exceeds LTV')
print('\n🇩🇪 Erkenntnis:')
print('   Champions      -> mit VIP-Treueprogramm schützen')
print('   At Risk        -> Rückgewinnungskampagne mit persönlichem Rabatt')
print('   Lost           -> deprioritisieren, Wiedergewinnungskosten übersteigen LTV')

## 9. Geographic Analysis / Geografische Analyse

In [ ]:
# EN Revenue and AOV by country / DE Umsatz und Ø-Bestellwert nach Land
geo = (
    df.groupby('Country')
    .agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'),
         Customers=('CustomerID','nunique'))
    .sort_values('Revenue', ascending=False)
    .head(15).reset_index()
)
geo['AOV']  = (geo['Revenue'] / geo['Orders']).round(2)
geo_intl    = geo[geo['Country'] != 'United Kingdom'].head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(geo_intl['Country'][::-1], geo_intl['Revenue'][::-1], color='teal', alpha=0.8)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x/1000:.0f}k'))
axes[0].set_title('Top 10 International Markets\nTop-10 internationale Märkte', fontweight='bold')

axes[1].scatter(geo_intl['Customers'], geo_intl['AOV'],
                s=geo_intl['Revenue']/500, alpha=0.7, color='steelblue')
for _, row in geo_intl.iterrows():
    axes[1].annotate(row['Country'], (row['Customers'], row['AOV']),
                     fontsize=8, textcoords='offset points', xytext=(4,4))
axes[1].set_xlabel('Number of Customers / Anzahl Kunden')
axes[1].set_ylabel('Avg Order Value / Ø Bestellwert (£)')
axes[1].set_title('Customers vs AOV (bubble = revenue)\nKunden vs. Ø-Bestellwert (Kreis = Umsatz)',
                  fontweight='bold')
plt.tight_layout()
plt.show()

top_intl = geo_intl.iloc[0]
print(f'\n🇬🇧 Insight: Largest international market: {top_intl["Country"]} (£{top_intl["Revenue"]:,.0f}).')
print('   High-AOV markets are prime B2B outreach targets.')
print(f'\n🇩🇪 Erkenntnis: Größter internationaler Markt: {top_intl["Country"]} (£{top_intl["Revenue"]:,.0f}).')
print('   Märkte mit hohem Ø-Bestellwert eignen sich ideal für B2B-Akquise.')

## 10. Statistical Analysis / Statistische Analyse

In [ ]:
# EN Order-level stats / DE Statistiken auf Bestellebene
order_level = df.groupby('InvoiceNo').agg(
    Revenue=('Revenue','sum'), Items=('Quantity','sum'), SKUs=('StockCode','nunique')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# EN Log-distribution / DE Log-Verteilung
axes[0].hist(np.log1p(order_level['Revenue']), bins=40,
             color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Order Value Distribution (log)\nBestellwertverteilung (log)', fontweight='bold')
axes[0].set_xlabel('log(Revenue + 1)')
axes[0].set_ylabel('Count / Anzahl')

# EN Items vs Revenue / DE Artikel vs. Umsatz
sample = order_level.sample(min(2000, len(order_level)), random_state=42)
axes[1].scatter(sample['Items'], sample['Revenue'], alpha=0.3, s=10, color='darkorange')
axes[1].set_xlabel('Items in Order / Artikel pro Bestellung')
axes[1].set_ylabel('Order Revenue / Bestellumsatz (£)')
axes[1].set_title('Items vs Revenue\nArtikel vs. Umsatz', fontweight='bold')
axes[1].set_xlim(0, sample['Items'].quantile(0.98))
axes[1].set_ylim(0, sample['Revenue'].quantile(0.98))

# EN RFM correlation / DE RFM-Korrelation
rfm_corr = rfm[['Recency','Frequency','Monetary']].corr()
sns.heatmap(rfm_corr, ax=axes[2], annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
axes[2].set_title('RFM Correlation Matrix\nRFM-Korrelationsmatrix', fontweight='bold')
plt.tight_layout()
plt.show()

corr_fm  = rfm['Frequency'].corr(rfm['Monetary'])
med_aov  = order_level['Revenue'].median()
mean_aov = order_level['Revenue'].mean()
print(f'\n🇬🇧 Insight: Frequency-Monetary correlation r={corr_fm:.2f}.')
print(f'   Median AOV £{med_aov:.2f} vs Mean AOV £{mean_aov:.2f} — right-skewed, use median.')
print(f'\n🇩🇪 Erkenntnis: Häufigkeit-Geldwert-Korrelation r={corr_fm:.2f}.')
print(f'   Median-Ø-Bestellwert £{med_aov:.2f} vs Durchschnitt £{mean_aov:.2f} — Median bevorzugen.')

## 11. Sales Forecast / Umsatzprognose

In [ ]:
# EN Linear trend forecast for next 3 months
# DE Lineare Trendprognose für die nächsten 3 Monate
monthly_full = (
    df[df['YearMonth'] < '2011-12']
    .groupby('YearMonth')['Revenue'].sum().reset_index()
)

x = np.arange(len(monthly_full))
y = monthly_full['Revenue'].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

x_fc       = np.arange(len(monthly_full), len(monthly_full) + 3)
y_fc       = slope * x_fc + intercept
fc_months  = ['2011-12', '2012-01', '2012-02']

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly_full['YearMonth'], y, color='steelblue', alpha=0.7,
       label='Actual / Tatsächlich')
ax.plot(monthly_full['YearMonth'], slope*x+intercept,
        color='red', lw=2, linestyle='--', label='Trend')
ax.bar(fc_months, y_fc, color='orange', alpha=0.85,
       label='Forecast / Prognose')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'£{v/1000:.0f}k'))
ax.set_title('Revenue Forecast — Linear Trend (next 3 months)\nUmsatzprognose — Linearer Trend (nächste 3 Monate)',
             fontweight='bold')
ax.set_xlabel('Month / Monat')
ax.set_ylabel('Revenue / Umsatz (£)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'\n🇬🇧 Forecast (R² = {r_value**2:.2f}):')
for m, f in zip(fc_months, y_fc):
    print(f'   {m}: £{f:,.0f}')
print('   Note: Linear model does not capture seasonality. Consider Prophet or SARIMA for production.')
print(f'\n🇩🇪 Prognose (R² = {r_value**2:.2f}):')
for m, f in zip(fc_months, y_fc):
    print(f'   {m}: £{f:,.0f}')
print('   Hinweis: Lineare Modelle erfassen keine Saisonalität. Prophet oder SARIMA für Produktion verwenden.')

## 12. Summary / Zusammenfassung

### 🇬🇧 Business Recommendations

| Priority | Finding | Recommended Action |
|---|---|---|
| 🔴 High | Month-1 retention ~25% | Launch 30-day post-purchase email sequence |
| 🔴 High | Champions drive disproportionate revenue | Create VIP loyalty programme |
| 🟡 Med | Clear Q4 peak | Pre-stock top SKUs by October; increase ad budget |
| 🟡 Med | At Risk segment exists | Win-back campaign with personalised 10% discount |
| 🟢 Low | High-AOV international markets | B2B outreach in Netherlands & EIRE |
| 🟢 Low | Tue-Thu morning is peak | Schedule email campaigns 9:00-11:00 |

### 🇩🇪 Geschäftsempfehlungen

| Priorität | Erkenntnis | Empfohlene Maßnahme |
|---|---|---|
| 🔴 Hoch | Monat-1-Bindungsrate ~25% | 30-Tage-E-Mail-Sequenz nach Kauf starten |
| 🔴 Hoch | Champions treiben überproportionalen Umsatz | VIP-Treueprogramm einführen |
| 🟡 Mittel | Klare Q4-Spitze | Top-SKUs bis Oktober bevorraten; Werbebudget erhöhen |
| 🟡 Mittel | Segment 'At Risk' vorhanden | Rückgewinnungskampagne mit 10% Rabatt |
| 🟢 Niedrig | Internationale Märkte mit hohem Ø-Bestellwert | B2B-Akquise in Niederlande & Irland |
| 🟢 Niedrig | Di-Do-Vormittag ist Spitzenzeit | E-Mail-Kampagnen 9:00-11:00 planen |

---
**🇬🇧 Conclusion:** Revenue follows an 80/20 rule across products and customers. Highest ROI actions: protect Champions, improve early retention.

**🇩🇪 Fazit:** Umsatz folgt der 80/20-Regel bei Produkten und Kunden. Höchste Rendite: Champions schützen und frühe Kundenbindung verbessern.